# Mixed precision: fp16, bf16 và loss scaling

Trên GPU từ đời Volta trở đi, phép nhân ma trận và convolution ở fp16/bf16 chạy trên **Tensor Core**, nhanh
gấp vài lần fp32, và activation lưu nửa số byte. Cái giá là phạm vi và độ chính xác của số: fp16 tràn ở 65 504
và làm tròn về 0 mọi thứ nhỏ hơn khoảng $$6 \times 10^{-8}$$, đúng vùng mà nhiều gradient nằm. Bạn đo điều đó trên
gradient thật, tự viết **dynamic loss scaling** khớp `torch.amp.GradScaler`, viết một bước train AMP đúng thứ
tự, rồi so fp32, fp16 và bf16 trên CIFAR-10.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm và lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, train, vẽ; test bỏ qua chúng.
- **Số liệu thật cần GPU**: chọn kernel `ai-gpu` (xem `setup/gpu/README.md`). Với env gốc (torch CPU) notebook vẫn
  chạy hết nhưng ở bản thu nhỏ (W=16, 10 000 ảnh, 2 epoch, chỉ fp32 và bf16; khoảng 3 phút) để kiểm code, không để đọc số.
- GPU 4GB: cả notebook dùng dưới 1 GB VRAM.
- Chấm: `uv run pytest tracks/ai-engineer/modules/02-pytorch-deep-dive/exercises/intermediate/01-mixed-precision -q`

In [ ]:
# explore: giới hạn luồng CPU, tìm gốc repo, kiểm CIFAR-10 (137 MB, tải một lần bằng env gốc)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import torch; GPU dùng chung, CPU cũng vậy
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
MODULE = ROOT / "tracks/ai-engineer/modules/02-pytorch-deep-dive"
CIFAR = ROOT / "data/raw/cifar10"
if not (CIFAR.is_dir() and any(CIFAR.iterdir())):
    # kernel ai-gpu không có thư viện tải dữ liệu: khi đó chạy ở terminal `uv run python data/download_dataset.py cifar10`
    sys.path.insert(0, str(ROOT))
    from data.download_dataset import ensure

    ensure("cifar10")
print(CIFAR, sorted(p.name for p in (CIFAR / "plain_text").iterdir()))

In [ ]:
import math
import time

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

## 1. Ép kiểu mất gì

`cast_report(t, dtype)` đo ba thứ khi ép một tensor fp32 về `dtype`, chỉ trên các phần tử hữu hạn, khác 0:
tỉ lệ **underflow** (thành 0), tỉ lệ **overflow** (thành ±inf), và **sai số tương đối trung vị** của phần còn
lại. So sánh ở float64 để phép đo không tự làm tròn.

In [ ]:
def cast_report(t: torch.Tensor, dtype: torch.dtype) -> dict[str, float]:
    """Ép t về dtype (fp16, bf16...) và đo mất mát, CHỈ trên các phần tử hữu hạn và khác 0 của t:

    - "underflow": tỉ lệ phần tử thành 0 sau khi ép;
    - "overflow": tỉ lệ phần tử thành ±inf;
    - "rel_err": trung vị của |ép - gốc| / |gốc| trên các phần tử không underflow, không overflow.

    Trả float Python; không còn phần tử nào để đo thì cả ba bằng 0.0. So sánh ở float64.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# explore: ba định dạng số, và ép thử vài giá trị điển hình
for dt in (torch.float32, torch.float16, torch.bfloat16):
    fi = torch.finfo(dt)
    print(f"{str(dt):15s} bits {fi.bits:2d} | max {fi.max:.3e} | normal nhỏ nhất {fi.tiny:.3e} | eps {fi.eps:.3e}")
samples = torch.tensor([1e-9, 1e-7, 1e-5, 1.0 + 2**-12, 3.14159, 7e4, 1e6])
for dt in (torch.float16, torch.bfloat16):
    print(dt, samples.to(dt).float().tolist(), cast_report(samples, dt))

## 2. Dynamic loss scaling tự viết

Nhân loss với $$S$$ thì mọi gradient nhân $$S$$ (chain rule tuyến tính), đẩy chúng ra khỏi vùng underflow của fp16.
Trước khi optimizer dùng, chia lại cho $$S$$ (ở fp32). Chọn $$S$$ **động**: gặp inf/NaN thì bỏ bước đó và giảm
$$S$$ một nửa; đủ `growth_interval` bước tốt liên tiếp thì tăng gấp đôi. `DynamicLossScaler` có cùng giao diện
`scale`, `unscale_`, `step`, `update`, `get_scale` với `torch.amp.GradScaler`, để `train_step` dùng được cả hai.

In [ ]:
class DynamicLossScaler:
    """Dynamic loss scaling tự viết, cùng giao diện và cùng hành vi với torch.amp.GradScaler:

    - scale(loss): trả loss * scale hiện tại.
    - unscale_(optimizer): chia .grad của mọi tham số trong optimizer cho scale (tại chỗ) và ghi nhận có
      phần tử inf/NaN nào không. Gọi lần hai trong cùng một bước thì không chia nữa.
    - step(optimizer): unscale_ nếu bước này chưa unscale; có inf/NaN thì BỎ QUA optimizer.step().
    - update(): có inf/NaN thì scale *= backoff_factor và đếm lại từ 0; không thì đếm thêm một bước tốt,
      đủ growth_interval bước tốt liên tiếp thì scale *= growth_factor và đếm lại. Rồi sang bước mới.
    - get_scale(): scale hiện tại (float).
    """

    def __init__(self, init_scale: float = 2.0**16, growth_factor: float = 2.0,
                 backoff_factor: float = 0.5, growth_interval: int = 2000):
        # TODO
        raise NotImplementedError

    def get_scale(self) -> float:
        # TODO
        raise NotImplementedError

    def scale(self, loss: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError

    def unscale_(self, optimizer: torch.optim.Optimizer) -> None:
        # TODO
        raise NotImplementedError

    def step(self, optimizer: torch.optim.Optimizer) -> None:
        # TODO
        raise NotImplementedError

    def update(self) -> None:
        # TODO
        raise NotImplementedError

## 3. Một bước train AMP

Thứ tự là chỗ hay sai: forward trong `autocast`, loss tính ở fp32, `backward` trên loss **đã scale**,
`unscale_` **trước** khi đo norm và clip (clip gradient đang bị nhân $$S$$ là clip sai ngưỡng $$S$$ lần), rồi
`step`, `update`. `stepped` cho biết scaler có bỏ bước này không.

In [ ]:
def train_step(model: nn.Module, x: torch.Tensor, y: torch.Tensor, optimizer: torch.optim.Optimizer,
               scaler, amp_dtype: torch.dtype | None, max_norm: float | None = None) -> dict:
    """Một bước train mixed precision. `scaler` là torch.amp.GradScaler (có thể enabled=False) hoặc
    DynamicLossScaler ở trên; amp_dtype None nghĩa là fp32 thuần.

    - Forward trong torch.autocast(device_type=x.device.type, dtype=amp_dtype) khi amp_dtype khác None;
      cross-entropy tính trên logits ép về float32.
    - Xoá gradient cũ, backward trên loss ĐÃ scale, unscale_ rồi mới đo norm gradient và clip
      (max_norm None thì không clip, vẫn đo), rồi scaler.step, scaler.update.
    - Trả {"loss": float, "grad_norm": float (norm TRƯỚC khi clip, đã unscale), "stepped": bool}; stepped
      False khi gradient có inf/NaN (scaler bỏ qua bước này).
    """
    # TODO
    raise NotImplementedError

In [ ]:
# explore: DynamicLossScaler của bạn và torch.amp.GradScaler trên cùng một chuỗi bước, có chèn gradient inf
def toy_run(scaler, steps=8, inf_at=(2, 6)):
    torch.manual_seed(0)
    lin = nn.Linear(4, 2)
    opt = torch.optim.SGD(lin.parameters(), lr=0.1)
    scales = []
    for t in range(steps):
        x = torch.randn(8, 4)
        if t in inf_at:
            x[0, 0] = float("inf")
        info = train_step(lin, x, torch.randint(0, 2, (8,)), opt, scaler, None)
        scales.append((scaler.get_scale(), info["stepped"]))
    return scales, lin.weight.detach().clone()


mine, w_mine = toy_run(DynamicLossScaler(init_scale=8.0, growth_interval=3))
ref, w_ref = toy_run(torch.amp.GradScaler("cpu", init_scale=8.0, growth_interval=3))
print("của bạn:", mine)
print("torch  :", ref)
print("trọng số khớp:", torch.allclose(w_mine, w_ref))

## 4. Gradient thật nằm ở đâu trên trục số

Cell dưới lấy gradient của ResNet-9 lúc khởi tạo trên một batch CIFAR-10 thật (tính fp32 trên CPU, có seed, nên
con số gần như giống nhau trên mọi máy) và hỏi: ép về fp16 thì bao nhiêu phần trăm thành 0, và nhân $$2^8$$, $$2^{16}$$
trước khi ép thì sao? Đo hai loại: gradient theo **trọng số** (thứ optimizer dùng, được cộng trên cả batch) và
gradient theo **activation** (thứ backward fp16 thật sự tính và truyền qua từng tầng, mỗi phần tử chỉ là phần của
một ảnh trong loss trung bình). bf16 có cùng phạm vi mũ với fp32 nên gần như không underflow, nhưng chỉ có 8 bit mantissa.

In [ ]:
# explore: dữ liệu và augmentation lấy từ data.py của bạn ở bài Basic 03 (chưa làm xong thì dùng lời giải mẫu)
import importlib.util


def load_basic03():
    probe = torch.zeros(2, 3, 32, 32, dtype=torch.uint8)
    for path in [MODULE / "exercises/basic/03-fast-dataloader/starter/data.py",
                 MODULE / "solutions/basic/03-fast-dataloader/data.py"]:
        spec = importlib.util.spec_from_file_location(f"basic03_{path.parent.name}", path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        try:
            mod.augment_batch(probe, torch.Generator().manual_seed(0))
            test = mod.decode_cifar_parquet(CIFAR / "plain_text/test-00000-of-00001.parquet")
            return mod, test, path
        except NotImplementedError:
            continue


data_mod, (x_test, y_test), used = load_basic03()
x_train, y_train = data_mod.decode_cifar_parquet(CIFAR / "plain_text/train-00000-of-00001.parquet")
print("data.py:", used.relative_to(ROOT))
print("train", tuple(x_train.shape), x_train.dtype, "| test", tuple(x_test.shape))

In [ ]:
# explore: ResNet nhỏ kiểu "ResNet-9" (DAWNBench), độ rộng W; cấu hình theo thiết bị
def conv_bn(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


class Residual(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.f = nn.Sequential(conv_bn(c, c), conv_bn(c, c))

    def forward(self, x):
        return x + self.f(x)


def resnet9(w=32, n_classes=10):
    return nn.Sequential(
        conv_bn(3, w),
        conv_bn(w, 2 * w), nn.MaxPool2d(2), Residual(2 * w),
        conv_bn(2 * w, 4 * w), nn.MaxPool2d(2),
        conv_bn(4 * w, 8 * w), nn.MaxPool2d(2), Residual(8 * w),
        nn.AdaptiveMaxPool2d(1), nn.Flatten(), nn.Linear(8 * w, n_classes),
    )


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":   # kernel ai-gpu: đủ CIFAR-10, khoảng 2 phút cho cả 4 cấu hình trên RTX 3050 4GB
    W, N_TRAIN, EPOCHS, BS = 32, 50_000, 3, 256
    MODES = ["fp32", "fp16 + GradScaler", "bf16", "fp16 không scaler", "fp16 + DynamicLossScaler"]
else:                  # env gốc (torch CPU): bản thu nhỏ để notebook chạy hết, số liệu thật cần GPU
    W, N_TRAIN, EPOCHS, BS = 16, 10_000, 2, 128
    MODES = ["fp32", "bf16"]
MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1).to(DEVICE)
STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1).to(DEVICE)
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| W", W, "| ảnh train", N_TRAIN,
      "| tham số", sum(p.numel() for p in resnet9(W).parameters()))

In [ ]:
# explore: gradient thật của ResNet-9 (W=32) lúc khởi tạo, một batch 256 ảnh train, tính fp32 trên CPU.
# Hai loại: gradient theo TRỌNG SỐ (p.grad) và gradient theo ACTIVATION (output của từng Conv2d, giữ bằng hook)
torch.manual_seed(0)
probe_model = resnet9(32)
acts = []


def keep_grad(module, inputs, output):
    output.retain_grad()
    acts.append(output)


hooks = [m.register_forward_hook(keep_grad) for m in probe_model.modules() if isinstance(m, nn.Conv2d)]
xb = (x_train[:256].float() / 255 - MEAN.cpu()) / STD.cpu()
F.cross_entropy(probe_model(xb), y_train[:256]).backward()
for h in hooks:
    h.remove()
weight_grads = torch.cat([p.grad.flatten() for p in probe_model.parameters()])
act_grads = torch.cat([a.grad.flatten() for a in acts])
for name, g in [("trọng số", weight_grads), ("activation", act_grads)]:
    print(f"gradient theo {name}: {g.numel():,} phần tử, |g| trung vị {g.abs().median():.2e}")
    for scale in (1.0, 2.0**8, 2.0**16):
        r16, rbf = cast_report(g * scale, torch.float16), cast_report(g * scale, torch.bfloat16)
        print(f"  scale {scale:>6.0f}: fp16 underflow {r16['underflow']:.4f} overflow {r16['overflow']:.4f}"
              f" rel_err {r16['rel_err']:.1e} | bf16 underflow {rbf['underflow']:.4f} rel_err {rbf['rel_err']:.1e}")
del acts, act_grads

## 5. fp32, fp16, bf16 trên CIFAR-10

Cùng seed, cùng thứ tự dữ liệu, cùng augmentation, SGD Nesterov + OneCycle, batch 256, 3 epoch. Dữ liệu uint8
nằm sẵn trên GPU và augmentation chạy theo batch trên GPU (bài Basic 03), nên thời gian đo là thời gian tính
chứ không phải thời gian đọc ảnh. Đo: giây/epoch, peak memory (`torch.cuda.max_memory_allocated`), test accuracy,
số bước scaler bỏ qua. Thêm hai cấu hình để so: fp16 **không** scaler, và fp16 với `DynamicLossScaler` của bạn.
GPU đang có người khác dùng thì thời gian dao động; chạy lại khi máy rảnh nếu số lạ.

In [ ]:
# explore: train ResNet-9 với từng chế độ số học, cùng seed và cùng thứ tự dữ liệu
def make_scaler(mode):
    if mode == "fp16 + GradScaler":
        return torch.amp.GradScaler(DEVICE)
    if mode == "fp16 + DynamicLossScaler":
        return DynamicLossScaler()
    return torch.amp.GradScaler(DEVICE, enabled=False)          # scale = 1, step bình thường


def evaluate(model, amp_dtype):
    model.eval()
    hits = 0
    with torch.no_grad(), torch.autocast(DEVICE, dtype=amp_dtype or torch.float32, enabled=amp_dtype is not None):
        for i in range(0, len(x_test), 1000):
            xb = ((x_test[i:i + 1000].to(DEVICE).float() / 255 - MEAN) / STD).contiguous(memory_format=torch.channels_last)
            hits += (model(xb).argmax(1).cpu() == y_test[i:i + 1000]).sum().item()
    return hits / len(x_test)


def run(mode):
    amp_dtype = None if mode == "fp32" else (torch.bfloat16 if mode == "bf16" else torch.float16)
    torch.manual_seed(0)
    gen = torch.Generator().manual_seed(0)
    model = resnet9(W).to(DEVICE).to(memory_format=torch.channels_last)
    opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, nesterov=True, weight_decay=5e-4)
    steps_per_epoch = N_TRAIN // BS
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 0.1, total_steps=EPOCHS * steps_per_epoch, pct_start=0.25)
    scaler = make_scaler(mode)
    xs, ys = x_train[:N_TRAIN].to(DEVICE), y_train[:N_TRAIN].to(DEVICE)   # cả tập uint8 nằm sẵn trên GPU: 150 MB
    if DEVICE == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    skipped, losses, start = 0, [], time.perf_counter()
    for epoch in range(EPOCHS):
        model.train()
        order = torch.randperm(N_TRAIN, generator=gen).to(DEVICE)
        for i in range(steps_per_epoch):
            idx = order[i * BS:(i + 1) * BS]
            xb = data_mod.augment_batch(xs[idx], gen)
            xb = ((xb.float() / 255 - MEAN) / STD).contiguous(memory_format=torch.channels_last)
            info = train_step(model, xb, ys[idx], opt, scaler, amp_dtype)
            skipped += not info["stepped"]
            losses.append(info["loss"])
            sched.step()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    peak = torch.cuda.max_memory_allocated() / 2**20 if DEVICE == "cuda" else float("nan")
    return {"mode": mode, "giây/epoch": seconds / EPOCHS, "peak MiB": peak, "test acc": evaluate(model, amp_dtype),
            "bước bị bỏ": skipped, "scale cuối": scaler.get_scale(), "loss cuối": float(np.mean(losses[-20:])),
            "losses": losses}


results = []
for mode in MODES:
    results.append(run(mode))
    r = results[-1]
    print(f"{mode:26s} {r['giây/epoch']:6.1f} s/epoch | peak {r['peak MiB']:6.0f} MiB | test acc {r['test acc']:.4f}"
          f" | bỏ {r['bước bị bỏ']} bước | scale cuối {r['scale cuối']:.0f} | loss 20 bước cuối {r['loss cuối']:.3f}")

In [ ]:
# explore: đường loss theo bước (trung bình trượt 20 bước)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
for r in results:
    smooth = np.convolve(r["losses"], np.ones(20) / 20, mode="valid")
    ax.plot(smooth, label=r["mode"])
ax.set_xlabel("bước"); ax.set_ylabel("train loss"); ax.legend(); plt.show()

## 6. Đọc kết quả

Trước khi làm trắc nghiệm trong app, tự trả lời:

1. Ở scale 1, phần gradient nào bị fp16 xoá? Vì sao nhân $$2^{16}$$ gần như cứu hết mà không làm tràn?
2. fp16 và bf16 nhanh hơn fp32 bao nhiêu lần, peak memory giảm bao nhiêu? Vì sao memory không giảm đúng một nửa?
3. fp16 không scaler vẫn train được ở đây. Điều đó có nghĩa là loss scaling thừa không? Khi nào nó thành bắt buộc?
4. Vì sao bf16 không cần scaler, và bạn trả giá bằng gì?